# IMU Expert — 结果存档

在你已经跑完 `train_imu.ipynb` 之后跑这个 notebook，它会：

1. **重新评估** `imu_classifier_best.pt`，拿到精确的 best acc 和 per-class 准确率
2. **备份权重**：把 `imu_expert.pt` 复制一份带版本+准确率标签的副本（防丢）
3. **写一份 `results.txt`**：汇总所有数字
4. **打包 zip**：把图、results.txt 都打包成一个文件让你下载，下载完丢进本地 repo 就能 commit

**前提**：先跑过 `train_imu.ipynb`，`pgmoe_ckpt/` 里有 `imu_classifier_best.pt` 和两张 png。

---

## 1. 挂载 Drive + 配置路径

`SAVE_DIR` 必须和 `train_imu.ipynb` 里一致——就是放 `.pt` 和 `.png` 的那个文件夹。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DATA_ROOT = "/content/drive/MyDrive/utd_mhad"
SAVE_DIR  = "/content/drive/MyDrive/pgmoe_ckpt"
BUNDLE_DIR = "/content/imu_archive"   # 临时打包目录，会被 zip

os.makedirs(BUNDLE_DIR, exist_ok=True)

# Sanity check
for fname in ["imu_classifier_best.pt", "imu_expert.pt",
              "training_curve.png", "confusion_matrix.png"]:
    p = os.path.join(SAVE_DIR, fname)
    print(f"{'✓' if os.path.exists(p) else '✗'} {fname}")

## 2. 重新构建模型 + 加载权重

这里要把 `train_imu.ipynb` 里的模型定义**重新写一遍**，因为 PyTorch 加载 `state_dict` 需要先有相同结构的网络实例。

（如果 cell 1 显示有 `✗`，说明 `pgmoe_ckpt/` 里缺文件，先回去把 `train_imu.ipynb` 跑完。）

In [ ]:
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMU_LEN = 192
NUM_CLASSES = 27
D_MODEL = 256
TEST_SUBJECTS = {2, 4, 6, 8}


class ResidualBlock1D(nn.Module):
    def __init__(self, in_c, out_c, kernel=5, stride=1):
        super().__init__()
        pad = kernel // 2
        self.conv = nn.Sequential(
            nn.Conv1d(in_c, out_c, kernel, stride=stride, padding=pad),
            nn.BatchNorm1d(out_c), nn.ReLU(inplace=True),
            nn.Conv1d(out_c, out_c, kernel, stride=1, padding=pad),
            nn.BatchNorm1d(out_c),
        )
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_c, out_c, 1, stride=stride),
                nn.BatchNorm1d(out_c),
            )
        else:
            self.shortcut = nn.Identity()
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.relu(self.conv(x) + self.shortcut(x))


class IMUExpert(nn.Module):
    def __init__(self, d_model=256, in_channels=6):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, 64, 7, stride=2, padding=3),
            nn.BatchNorm1d(64), nn.ReLU(inplace=True),
        )
        self.block1 = ResidualBlock1D(64, 128, 5, 2)
        self.block2 = ResidualBlock1D(128, 256, 5, 2)
        self.block3 = ResidualBlock1D(256, d_model, 3, 2)
    def forward(self, x):
        x = self.stem(x); x = self.block1(x); x = self.block2(x); x = self.block3(x)
        return x.transpose(1, 2)


class IMUClassifier(nn.Module):
    def __init__(self, num_classes=27, d_model=256):
        super().__init__()
        self.encoder = IMUExpert(d_model=d_model)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)
    def forward(self, x):
        t = self.encoder(x); p = t.mean(dim=1)
        return self.head(self.norm(p))


model = IMUClassifier().to(device)
model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "imu_classifier_best.pt"),
                                  map_location=device))
model.eval()
print("Model loaded, params:", f"{sum(p.numel() for p in model.parameters()):,}")

## 3. 重建 test set + 评估

Dataset 类也要重新定义。这里是测试集 only（subjects 2/4/6/8）。

In [ ]:
def parse_filename(fname):
    parts = fname.split("_")
    return int(parts[0][1:]), int(parts[1][1:]), int(parts[2][1:])


class IMUDataset(Dataset):
    def __init__(self, data_root, subjects):
        self.samples = []
        folder = os.path.join(data_root, "Inertial")
        for fname in sorted(os.listdir(folder)):
            if not fname.endswith("_inertial.mat"): continue
            a, s, _ = parse_filename(fname)
            if s not in subjects: continue
            self.samples.append((os.path.join(folder, fname), a - 1))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        fpath, label = self.samples[idx]
        data = sio.loadmat(fpath)["d_iner"].astype(np.float32)
        if data.shape[0] < IMU_LEN:
            data = np.concatenate([data, np.zeros((IMU_LEN - data.shape[0], 6), np.float32)], 0)
        x = torch.from_numpy(data[:IMU_LEN]).T.contiguous()
        return x, label


test_ds = IMUDataset(DATA_ROOT, TEST_SUBJECTS)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

preds, labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        preds.extend(model(x.to(device)).argmax(1).cpu().numpy())
        labels.extend(y.numpy())
preds, labels = np.array(preds), np.array(labels)

best_acc = float(accuracy_score(labels, preds))
print(f"Test accuracy: {best_acc:.4f}  ({best_acc*100:.2f}%)")
print(f"Test samples : {len(labels)}")
print(f"Midterm base : 67.90%   delta: {(best_acc-0.679)*100:+.2f}%")

## 4. Per-class 准确率（找最差的 3 个类）

最差的几个类暴露出 IMU 模态的弱点——后面 phase arbitrator 要让 vision 在这些类上承担更多权重。

UTD-MHAD 27 类（label 0–26）大致是：右臂挥拳、横扫左、横扫右、拍手、扔、叉手等动作。

In [ ]:
UTD_LABELS = [
    "right arm swipe to the left", "right arm swipe to the right",
    "right hand wave", "two hand front clap", "right arm throw",
    "cross arms in the chest", "basketball shooting", "right hand draw x",
    "right hand draw circle CW", "right hand draw circle CCW",
    "draw triangle", "bowling", "front boxing", "baseball swing",
    "tennis right hand forehand", "arm curl", "tennis serve",
    "two hand push", "right hand knock", "right hand catch",
    "right hand pickup and throw", "jogging", "walking",
    "sit to stand", "stand to sit", "forward lunge", "squat",
]

per_class = []
for c in range(NUM_CLASSES):
    mask = labels == c
    if mask.sum() == 0: continue
    per_class.append((c, float((preds[mask] == c).mean()), int(mask.sum())))

per_class.sort(key=lambda r: r[1])
print("Per-class accuracy (sorted ascending, worst first):\n")
for c, acc_c, n in per_class:
    bar = "#" * int(acc_c * 30)
    name = UTD_LABELS[c] if c < len(UTD_LABELS) else f"class {c}"
    print(f"  c{c:2d}  acc {acc_c:.3f}  n={n:2d}  | {bar:<30}  {name}")

worst_3 = per_class[:3]
print("\nWorst 3 classes (these benefit most from vision):")
for c, acc_c, n in worst_3:
    print(f"  c{c}: {UTD_LABELS[c]}  acc={acc_c:.3f}")

## 5. 备份权重 + 复制图 + 写 results.txt

所有产物会先汇总到 `/content/imu_archive/`，然后下一步打包成 zip。

权重备份用的命名格式：`imu_expert_v1_acc{XX.XX}.pt`，**用准确率小数点替换成下划线**（文件名不能有点号会被解析成扩展名）。

In [ ]:
import shutil

acc_tag = f"{best_acc*100:.2f}".replace(".", "_")

# 1) 备份权重（带版本+acc标签）到 Drive
backup_dir = os.path.join(SAVE_DIR, "backups")
os.makedirs(backup_dir, exist_ok=True)
for src_name in ["imu_expert.pt", "imu_classifier_best.pt"]:
    src = os.path.join(SAVE_DIR, src_name)
    dst = os.path.join(backup_dir, src_name.replace(".pt", f"_v1_acc{acc_tag}.pt"))
    shutil.copy(src, dst)
    print(f"backup → {dst}")

# 2) 复制图到 bundle 目录（要进 git）
for src_name, dst_name in [
    ("training_curve.png",   "imu_training_curve.png"),
    ("confusion_matrix.png", "imu_confusion_matrix.png"),
]:
    shutil.copy(os.path.join(SAVE_DIR, src_name),
                os.path.join(BUNDLE_DIR, dst_name))
    print(f"bundle ← {dst_name}")

# 3) 写 results.txt
lines = []
lines.append("# IMU Expert Results\n")
lines.append(f"Test accuracy:    {best_acc:.4f}  ({best_acc*100:.2f}%)\n")
lines.append(f"Test samples:     {len(labels)}\n")
lines.append(f"Midterm baseline: 0.6790  (67.90%)\n")
lines.append(f"Delta:            {(best_acc-0.679)*100:+.2f} percentage points\n")
lines.append("\n## Per-class accuracy (sorted ascending)\n")
for c, acc_c, n in per_class:
    name = UTD_LABELS[c] if c < len(UTD_LABELS) else f"class {c}"
    lines.append(f"  c{c:2d}  acc={acc_c:.3f}  n={n:2d}  {name}\n")
lines.append("\n## Worst 3 classes (these will benefit most from vision fusion)\n")
for c, acc_c, n in worst_3:
    name = UTD_LABELS[c] if c < len(UTD_LABELS) else f"class {c}"
    lines.append(f"  c{c}: {name}  acc={acc_c:.3f}\n")

results_path = os.path.join(BUNDLE_DIR, "imu_results.txt")
with open(results_path, "w") as f:
    f.writelines(lines)
print(f"wrote → {results_path}")

print("\nbundle contents:")
for f in sorted(os.listdir(BUNDLE_DIR)):
    p = os.path.join(BUNDLE_DIR, f)
    sz = os.path.getsize(p)
    print(f"  {f}  ({sz/1024:.1f} KB)")

## 6. 打包 + 下载

把 `imu_archive/` 整个打成一个 zip 文件，然后调用 `files.download` 让浏览器弹出下载窗口。

**下载完之后**：在你本地电脑解压，把里面 3 个文件分别放到：
- `imu_training_curve.png`   →  `project/final/figures/`
- `imu_confusion_matrix.png` →  `project/final/figures/`
- `imu_results.txt`          →  `project/final/figures/`  (或 `project/final/report/`)

然后告诉 Claude（我），我帮你 commit 进 git。

In [ ]:
from google.colab import files

zip_path = "/content/imu_archive.zip"
shutil.make_archive(zip_path.replace(".zip", ""), "zip", BUNDLE_DIR)
print(f"zipped → {zip_path}  ({os.path.getsize(zip_path)/1024:.1f} KB)")

files.download(zip_path)

## 完事

**已经做完的事：**
- ✓ `pgmoe_ckpt/backups/` 里有了带版本+acc 标签的权重备份（防丢）
- ✓ 浏览器下载了 `imu_archive.zip`（含两张图 + results.txt）
- ✓ 精确 acc 数字和最差 3 类已经打印出来了（截图发我或读给我听）

**你接下来做：**
1. 把 zip 解压到本地 repo 的 `project/final/figures/`
2. 告诉我 acc 数字和最差 3 个类
3. 我帮你更新 devlog 并 commit 到 git
4. 然后开始写 `phase_arbitrator.py`